In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import warnings
warnings.filterwarnings('ignore')
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import tensorflow as tf
import keras
from keras import layers
from keras.applications import efficientnet
from tensorflow.keras.utils import to_categorical, plot_model
from keras.layers import TextVectorization
from keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import corpus_bleu
from tqdm import tqdm_notebook
from collections import Counter

In [2]:
IMAGES_PATH = r"C:\Users\hello\Downloads\MinorProject\flikr8k\Images"
CAPTIONS_PATH = r"C:\Users\hello\Downloads\MinorProject\flikr8k\captions.txt"
IMAGE_SIZE = (299, 299)
SEQ_LENGTH = 25
VOCAB_SIZE = 10000
EMBED_DIM = 512
FF_DIM = 512
BATCH_SIZE = 64
EPOCHS = 30


In [ ]:
def load_captions_data(filename):
    with open(filename) as caption_file:
        caption_data = caption_file.readlines()[1:]
        caption_mapping = {}
        text_data = []
        images_to_skip = set()

        for line in caption_data:
            line = line.rstrip("\n")

            img_name, caption = line.split(",", 1)
            img_name = os.path.join(IMAGES_PATH, img_name.strip())

            tokens = caption.strip().split()
            if len(tokens) < 5 or len(tokens) > SEQ_LENGTH:
                images_to_skip.add(img_name)
                continue

            if img_name.endswith("jpg") and img_name not in images_to_skip:
                caption = "<start> " + caption.strip() + " <end>"
                text_data.append(caption)

                if img_name in caption_mapping:
                    caption_mapping[img_name].append(caption)
                else:
                    caption_mapping[img_name] = [caption]

        for img_name in images_to_skip:
            if img_name in caption_mapping:
                del caption_mapping[img_name]

        return caption_mapping, text_data

def train_val_split(caption_data, validation_size=0.2, test_size=0.05, shuffle=True):

    all_images = list(caption_data.keys())
    
    if shuffle:
        np.random.shuffle(all_images)
    
    train_keys, validation_keys = train_test_split(all_images, test_size=validation_size, random_state=42)
    validation_keys, test_keys = train_test_split(validation_keys, test_size=test_size, random_state=42)
    
    training_data = {img_name: caption_data[img_name] for img_name in train_keys}
    validation_data = {img_name: caption_data[img_name] for img_name in validation_keys}
    test_data = {img_name: caption_data[img_name] for img_name in test_keys}


    return training_data, validation_data, test_data


captions_mapping, text_data = load_captions_data(CAPTIONS_PATH)


train_data, validation_data, test_data = train_val_split(captions_mapping)
print(f"Total number of samples: {len(captions_mapping)}")
print(f"----> Number of training samples: {len(train_data)}")
print(f"----> Number of validation samples: {len(validation_data)}")
print(f"----> Number of test samples: {len(test_data)}")

In [ ]:
def custom_standardization(input_string):

    lowercase = tf.strings.lower(input_string)

    strip_chars = "!\"#$%&'()*+,-./:;=?@[\]^_`{|}~1234567890"
    return tf.strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")


vectorization = TextVectorization(

    max_tokens=VOCAB_SIZE,
    output_mode="int",

    output_sequence_length=SEQ_LENGTH,

    standardize=custom_standardization)


vectorization.adapt(text_data)


image_augmentation = keras.Sequential([layers.RandomFlip("horizontal"),
                                       layers.RandomRotation(0.2),
                                       layers.RandomContrast(0.3)])


text_data = list(map(lambda x: str(custom_standardization(x).numpy())[2:-1], text_data))

In [ ]:
def visualaization(data, num_of_images):
    count = 1
    fig = plt.figure(figsize=(10,20))
    for filename in list(data.keys())[100:100+num_of_images]:
        captions = list(map(lambda x: str(custom_standardization(x).numpy())[2:-1], data[filename]))
        image_load = load_img(filename, target_size=(199,199,3))

        ax = fig.add_subplot(num_of_images,2,count,xticks=[],yticks=[])
        ax.imshow(image_load)
        count += 1

        ax = fig.add_subplot(num_of_images,2,count)
        plt.axis('off')
        ax.plot()
        ax.set_xlim(0,1)
        ax.set_ylim(0,len(captions))
        for i, caption in enumerate(captions):
            ax.text(0,i,caption,fontsize=20)
        count += 1
    plt.show()
    
visualaization(train_data, 7)

In [ ]:
def captions_length(data):
    plt.figure(figsize=(15, 7), dpi=300)
    sns.set_style('darkgrid')
    sns.histplot(x=[len(x.split(' ')) for x in data], kde=True, binwidth=1) 
    plt.title('Captions length histogram', fontsize=15, fontweight='bold')
    plt.xticks(fontweight='bold')
    plt.yticks(fontweight='bold')
    plt.xlabel('Length', fontweight='bold')
    plt.ylabel('Freaquency', fontweight='bold')
    plt.show()
    
captions_length(text_data)

In [ ]:
def word_occurrences(data):

    all_text = ' '.join(data)
    all_text = all_text.replace('a ', '')
    all_text = all_text.replace('<start> ', '')
    all_text = all_text.replace('<end> ', '')

    word_counts = Counter(all_text.split())

    words = list(word_counts.keys())[:30]
    values = list(word_counts.values())[:30]


    normalized_values = np.array(values) / np.max(values)
    colors = np.array(['rgba(30, 58, 138, {})'.format(0.4 + 0.5 * (value)) for value in normalized_values])

    fig = go.Figure(data=[go.Pie(labels=words, values=values, hole=.6, marker=dict(colors=colors), textinfo='label')])

    fig.update_layout(title_text='Word occurrences in captions (except for letter \'a\')', title_font=dict(size=23, family='Balto'))

    fig.show()
    
word_occurrences(text_data)

In [ ]:

def decode_and_resize(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img


def process_input(img_path, captions):

    return decode_and_resize(img_path), vectorization(captions)


def make_dataset(images, captions):
    dataset = tf.data.Dataset.from_tensor_slices((images, captions))
    dataset = dataset.shuffle(BATCH_SIZE * 8)
    dataset = dataset.map(process_input, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return dataset



train_dataset = make_dataset(list(train_data.keys()), list(train_data.values()))
validation_dataset = make_dataset(list(validation_data.keys()), list(validation_data.values()))

In [ ]:
def get_cnn_model():
    base_model = efficientnet.EfficientNetB0(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False, 
        weights="imagenet")

    base_model.trainable = False
    base_model_out = base_model.output
    base_model_out = layers.Reshape((-1, base_model_out.shape[-1]))(base_model_out)
    cnn_model = keras.models.Model(base_model.input, base_model_out)
    return cnn_model


class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.0)
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.dense_1 = layers.Dense(embed_dim, activation="relu")

    def call(self, inputs, training, mask=None):
        inputs = self.layernorm_1(inputs)
        inputs = self.dense_1(inputs)
        attention_output_1 = self.attention_1(query=inputs,
                                              value=inputs,
                                              key=inputs,
                                              attention_mask=None,
                                              training=training)
        out_1 = self.layernorm_2(inputs + attention_output_1)
        return out_1


class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.embed_scale = tf.math.sqrt(tf.cast(embed_dim, tf.float32))

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1) # Positional encoding
        embedded_tokens = self.token_embeddings(inputs) # Input embedding
        embedded_tokens = embedded_tokens * self.embed_scale
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions # Positional embedding

    def compute_mask(self, inputs, mask=None):
        return tf.math.not_equal(inputs, 0)


class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, ff_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.cross_attention_2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.ffn_layer_1 = layers.Dense(ff_dim, activation="relu")
        self.ffn_layer_2 = layers.Dense(embed_dim)

        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()

        self.embedding = PositionalEmbedding(embed_dim=EMBED_DIM,
                                             sequence_length=SEQ_LENGTH,
                                             vocab_size=VOCAB_SIZE,)
        self.out = layers.Dense(VOCAB_SIZE, activation="softmax")

        self.dropout_1 = layers.Dropout(0.3)
        self.dropout_2 = layers.Dropout(0.5)
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, training, mask=None):
        inputs = self.embedding(inputs)
        causal_mask = self.get_causal_attention_mask(inputs)
        

        if mask is not None:
            padding_mask = tf.cast(mask[:, :, tf.newaxis], dtype=tf.int32)
            combined_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32)

            combined_mask = tf.minimum(combined_mask, causal_mask)

        attention_output_1 = self.attention_1(query=inputs,
                                              value=inputs,
                                              key=inputs,
                                              attention_mask=combined_mask,
                                              training=training)
        out_1 = self.layernorm_1(inputs + attention_output_1)
        

        cross_attention_output_2 = self.cross_attention_2(query=out_1,
                                              value=encoder_outputs,
                                              key=encoder_outputs,
                                              attention_mask=padding_mask,
                                              training=training)
        out_2 = self.layernorm_2(out_1 + cross_attention_output_2)

        ffn_out = self.ffn_layer_1(out_2)
        ffn_out = self.dropout_1(ffn_out, training=training)
        ffn_out = self.ffn_layer_2(ffn_out)

        ffn_out = self.layernorm_3(ffn_out + out_2, training=training)
        ffn_out = self.dropout_2(ffn_out, training=training)
        
        preds = self.out(ffn_out)
        return preds
    

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = tf.concat([tf.expand_dims(batch_size, -1),tf.constant([1, 1], dtype=tf.int32)],axis=0)
        return tf.tile(mask, mult)


class ImageCaptioningModel(keras.Model):
    def __init__(self, cnn_model, encoder, decoder, num_captions_per_image=5, image_aug=None):
        super().__init__()
        self.cnn_model = cnn_model
        self.encoder = encoder
        self.decoder = decoder
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.acc_tracker = keras.metrics.Mean(name="accuracy")
        self.num_captions_per_image = num_captions_per_image
        self.image_aug = image_aug
        
        print()
        print(f'CNN input shape: {cnn_model.input_shape}')
        print(f'CNN output shape: {cnn_model.output_shape}', end='\n'*2)
        print(f'Encoder input ---> Dense layer shape: {cnn_model.output_shape} ---> (None, {cnn_model.output_shape[1]}, {EMBED_DIM})')
        print(f'Encoder output shape: (None, {cnn_model.output_shape[1]}, {EMBED_DIM})', end='\n'*2)
        print(f'Decoder input 1 (Caption) ---> Positional Embedding shape: (None, {SEQ_LENGTH-1}) ---> (None, {SEQ_LENGTH-1}, {EMBED_DIM})')
        print(f'Decoder input 2 (Embedded image features) shape: (None, {cnn_model.output_shape[1]}, {EMBED_DIM})')
        print(f'Decoder output (MH Cross-Attention) shape: (None, {SEQ_LENGTH-1}, {EMBED_DIM})')
        print(f'Decoder prediction (Dense layer) shape: (None, {SEQ_LENGTH-1}, {VOCAB_SIZE})')
        
    

    def calculate_loss(self, y_true, y_pred, mask):
        loss = self.loss(y_true, y_pred)
        mask = tf.cast(mask, dtype=loss.dtype)
        loss *= mask
        return tf.reduce_sum(loss) / tf.reduce_sum(mask)
    

    def calculate_accuracy(self, y_true, y_pred, mask):
        accuracy = tf.equal(y_true, tf.argmax(y_pred, axis=2))
        accuracy = tf.math.logical_and(mask, accuracy)
        accuracy = tf.cast(accuracy, dtype=tf.float32)
        mask = tf.cast(mask, dtype=tf.float32)
        return tf.reduce_sum(accuracy) / tf.reduce_sum(mask)

    def _compute_caption_loss_and_acc(self, img_embed, batch_seq, training=True):
        encoder_out = self.encoder(img_embed, training=training)
        batch_seq_inp = batch_seq[:, :-1]
        batch_seq_true = batch_seq[:, 1:]

        mask = tf.math.not_equal(batch_seq_true, 0)
        batch_seq_pred = self.decoder(batch_seq_inp, encoder_out, training=training, mask=mask)
        loss = self.calculate_loss(batch_seq_true, batch_seq_pred, mask)
        acc = self.calculate_accuracy(batch_seq_true, batch_seq_pred, mask)
        return loss, acc
    

    def train_step(self, batch_data):
        batch_img, batch_seq = batch_data
        batch_loss = 0
        batch_acc = 0
        

        if self.image_aug:
            batch_img = self.image_aug(batch_img)


        img_embed = self.cnn_model(batch_img)


        for i in range(self.num_captions_per_image):
            with tf.GradientTape() as tape:
                loss, acc = self._compute_caption_loss_and_acc(img_embed, batch_seq[:, i, :], training=True)


                batch_loss += loss
                batch_acc += acc


            train_vars = (self.encoder.trainable_variables + self.decoder.trainable_variables)


            grads = tape.gradient(loss, train_vars)


            self.optimizer.apply_gradients(zip(grads, train_vars))


        batch_acc /= float(self.num_captions_per_image)
        self.loss_tracker.update_state(batch_loss)
        self.acc_tracker.update_state(batch_acc)


        return {"loss": self.loss_tracker.result(),
                "acc": self.acc_tracker.result()}
    

    def test_step(self, batch_data):
        batch_img, batch_seq = batch_data
        batch_loss = 0
        batch_acc = 0


        img_embed = self.cnn_model(batch_img)


        for i in range(self.num_captions_per_image):
            loss, acc = self._compute_caption_loss_and_acc(img_embed, batch_seq[:, i, :], training=False)


            batch_loss += loss
            batch_acc += acc

        batch_acc /= float(self.num_captions_per_image)


        self.loss_tracker.update_state(batch_loss)
        self.acc_tracker.update_state(batch_acc)


        return {"loss": self.loss_tracker.result(),
                "acc": self.acc_tracker.result()}

    @property
    def metrics(self):

        return [self.loss_tracker, self.acc_tracker]


cnn_model = get_cnn_model()
encoder = TransformerEncoderBlock(embed_dim=EMBED_DIM, dense_dim=FF_DIM, num_heads=2)
decoder = TransformerDecoderBlock(embed_dim=EMBED_DIM, ff_dim=FF_DIM, num_heads=3)
caption_model = ImageCaptioningModel(cnn_model=cnn_model, encoder=encoder, decoder=decoder, image_aug=image_augmentation)

In [ ]:
cross_entropy = keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')


early_stopping = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)



class LRSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, post_warmup_learning_rate, warmup_steps):
        super().__init__()
        self.post_warmup_learning_rate = post_warmup_learning_rate
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        global_step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        warmup_progress = global_step / warmup_steps
        warmup_learning_rate = self.post_warmup_learning_rate * warmup_progress
        return tf.cond(
            global_step < warmup_steps,
            lambda: warmup_learning_rate,
            lambda: self.post_warmup_learning_rate)
    

num_train_steps = len(train_dataset) * EPOCHS
num_warmup_steps = num_train_steps // 15
lr_schedule = LRSchedule(post_warmup_learning_rate=1e-4, warmup_steps=num_warmup_steps)


caption_model.compile(optimizer=keras.optimizers.Adam(lr_schedule), loss=cross_entropy)


In [ ]:
import pickle
from tensorflow.keras.layers import TextVectorization


from_disk = pickle.load(open(r"C:\Users\hello\Downloads\MinorProject\tv_layer.pkl", "rb"))
cfg, wts = from_disk["config"], from_disk["weights"]


vocab_list = list(wts[0])


new_v = TextVectorization(
    max_tokens=cfg["max_tokens"],
    output_mode=cfg["output_mode"],
    ngrams=cfg["ngrams"],
    standardize=custom_standardization,     
    split=cfg["split"],
    output_sequence_length=cfg["output_sequence_length"],
    pad_to_max_tokens=cfg.get("pad_to_max_tokens", False),
    vocabulary=vocab_list              
)


print(new_v("this"))

In [ ]:
cnn_model = get_cnn_model()
encoder = TransformerEncoderBlock(embed_dim=EMBED_DIM, dense_dim=FF_DIM, num_heads=2)
decoder = TransformerDecoderBlock(embed_dim=EMBED_DIM, ff_dim=FF_DIM, num_heads=3)
mm1 = ImageCaptioningModel(cnn_model=cnn_model, encoder=encoder, decoder=decoder, image_aug=image_augmentation)

In [ ]:

cross_entropy = keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')


early_stopping = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)


class LRSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, post_warmup_learning_rate, warmup_steps):
        super().__init__()
        self.post_warmup_learning_rate = post_warmup_learning_rate
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        global_step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        warmup_progress = global_step / warmup_steps
        warmup_learning_rate = self.post_warmup_learning_rate * warmup_progress
        return tf.cond(
            global_step < warmup_steps,
            lambda: warmup_learning_rate,
            lambda: self.post_warmup_learning_rate)
    

num_train_steps = len(train_dataset) * EPOCHS
num_warmup_steps = num_train_steps // 15
lr_schedule = LRSchedule(post_warmup_learning_rate=1e-4, warmup_steps=num_warmup_steps)


mm1.compile(optimizer=keras.optimizers.Adam(lr_schedule), loss=cross_entropy)


gg = mm1.fit(train_dataset, epochs=2, validation_data=validation_dataset, callbacks=[early_stopping])

In [ ]:
mm1.load_weights(r"C:\Users\hello\Downloads\MinorProject\model_weights.h5")

In [ ]:
vocab = new_v.get_vocabulary()
INDEX_TO_WORD = {idx: word for idx, word in enumerate(vocab)}
MAX_DECODED_SENTENCE_LENGTH = SEQ_LENGTH - 1
test_images = list(test_data.keys())

def greedy_algorithm(image):

    image = decode_and_resize(image)


    image = tf.expand_dims(image, 0)
    image = mm1.cnn_model(image)


    encoded_img = mm1.encoder(image, training=False)


    decoded_caption = "<start> "
    for i in range(MAX_DECODED_SENTENCE_LENGTH):
        tokenized_caption = new_v([decoded_caption])[:, :-1]
        mask = tf.math.not_equal(tokenized_caption, 0)
        predictions = mm1.decoder(tokenized_caption, encoded_img, training=False, mask=mask)
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = INDEX_TO_WORD[sampled_token_index]
        if sampled_token == "<end>":
            break
        decoded_caption += " " + sampled_token

    decoded_caption = decoded_caption.replace("<start> ", "")
    decoded_caption = decoded_caption.replace(" <end>", "").strip()
    
    return decoded_caption

In [ ]:
vocab = new_v.get_vocabulary()
INDEX_TO_WORD = {idx: word for idx, word in enumerate(vocab)}
MAX_DECODED_SENTENCE_LENGTH = SEQ_LENGTH - 1
test_images = list(test_data.keys())

def greedy_algorithm(image):

    image = decode_and_resize(image)


    image = tf.expand_dims(image, 0)
    image = mm1.cnn_model(image)


    encoded_img = mm1.encoder(image, training=False)


    decoded_caption = "<start> "
    for i in range(MAX_DECODED_SENTENCE_LENGTH):
        tokenized_caption = new_v([decoded_caption])[:, :-1]
        mask = tf.math.not_equal(tokenized_caption, 0)
        predictions = mm1.decoder(tokenized_caption, encoded_img, training=False, mask=mask)
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = INDEX_TO_WORD[sampled_token_index]
        if sampled_token == "<end>":
            break
        decoded_caption += " " + sampled_token

    decoded_caption = decoded_caption.replace("<start> ", "")
    decoded_caption = decoded_caption.replace(" <end>", "").strip()
    
    return decoded_caption

In [ ]:

generated_captions = {}

pbar = tqdm_notebook(total=len(test_data), position=0, leave=True, colour='green')
for image_id in test_data:
    cap = greedy_algorithm(image_id)
    generated_captions[image_id] = cap
    pbar.update(1)
    
pbar.close()

In [ ]:

def BLEU_score(actual, predicted):

    processed_actual = []
    for i in actual:
        cap = [INDEX_TO_WORD[x] for x in vectorization(i).numpy() if INDEX_TO_WORD[x] != '']
        cap = ' '.join(cap)
        processed_actual.append(cap)
    

    b1=corpus_bleu(processed_actual, predicted, weights=(1.0, 0, 0, 0))
    b2=corpus_bleu(processed_actual, predicted, weights=(0.5, 0.5, 0, 0))
    b3=corpus_bleu(processed_actual, predicted, weights=(0.3, 0.3, 0.3, 0))
    b4=corpus_bleu(processed_actual, predicted, weights=(0.25, 0.25, 0.25, 0.25))
    
    return [
        (f'BLEU-4: {round(b4, 5)}'),
        (f'BLEU-3: {round(b3, 5)}'),
        (f'BLEU-2: {round(b2, 5)}'),
        (f'BLEU-1: {round(b1, 5)}'),
        (f'Generated Caption: {predicted[0]}'),
    ]

In [ ]:
def visualization(data, generated_captions, evaluator, num_of_images):
    keys = list(data.keys()) 
    images = [np.random.choice(keys) for i in range(num_of_images)] 
    
    count = 1
    fig = plt.figure(figsize=(6,20))    
    for filename in images:
        actual_cap = data[filename]
        actual_cap = [x.replace("<start> ", "") for x in actual_cap] 
        actual_cap = [x.replace(" <end>", "") for x in actual_cap] 
        
        caption = generated_captions[filename]
        caps_with_score = evaluator(actual_cap, [caption]*(len(actual_cap)))
    
        image_load = load_img(filename, target_size=(199,199,3))
        ax = fig.add_subplot(num_of_images,2,count,xticks=[],yticks=[])
        ax.imshow(image_load)
        count += 1

        ax = fig.add_subplot(num_of_images,2,count)
        plt.axis('off')
        ax.plot()
        ax.set_xlim(0,1)
        ax.set_ylim(0,len(caps_with_score))
        for i, text in enumerate(caps_with_score):
            ax.text(0,i,text,fontsize=10)
        count += 1
    plt.show()
    
visualization(test_data, generated_captions, BLEU_score, 7)

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk   

class CaptionApp:
    def __init__(self, master):
        self.master      = master
        master.title("Image-to-Caption")

       
        self.preview_lbl = tk.Label(master)
        self.preview_lbl.pack(padx=10, pady=10)

       
        self.caption_var = tk.StringVar()
        cap_frame = tk.Frame(master)
        cap_frame.pack(fill="x", padx=10, pady=(0, 10))
        tk.Label(cap_frame, text="Caption:").pack(side="left", anchor="w")
        tk.Entry(cap_frame, textvariable=self.caption_var,
                 state="readonly", width=80).pack(side="left",
                                                  fill="x", expand=True)

       
        btn_frame = tk.Frame(master)
        btn_frame.pack(pady=5)
        tk.Button(btn_frame, text="Open image", command=self.open_image
                 ).pack(side="left", padx=5)
        tk.Button(btn_frame, text="Generate caption", command=self.gen_caption
                 ).pack(side="left", padx=5)

        self.path   = None         
        self.tk_img = None          

    
    def open_image(self):
        ft = [("Images", "*.jpg *.jpeg *.png *.bmp *.gif"), ("All files", "*.*")]
        path = filedialog.askopenfilename(title="Choose an image", filetypes=ft)
        if not path:
            return
        self.path = path
        try:
    
            thumb      = Image.open(path).resize((299, 299))
            self.tk_img = ImageTk.PhotoImage(thumb)
            self.preview_lbl.config(image=self.tk_img)
            self.caption_var.set("")           
        except Exception as e:
            messagebox.showerror("Error", str(e))
            self.path = None

    def gen_caption(self):
        if not self.path:
            messagebox.showinfo("Info", "Please select an image first.")
            return
        try:
            caption = greedy_algorithm(self.path)   
            
            self.caption_var.set(caption)
        except Exception as e:
            messagebox.showerror("Error", str(e))


if __name__ == "__main__":
    root = tk.Tk()
    CaptionApp(root)
    root.mainloop()


In [ ]:
import tkinter as tk
from tkinter import filedialog, ttk
from PIL import Image, ImageTk
import numpy as np

selected_image_path_gui = None 
image_display_label_gui = None
caption_display_label_gui = None
progress_bar_gui = None
root_gui = None

def display_image_gui(image_path):
    """Loads and displays the selected image in the GUI."""
    global image_display_label_gui
    try:
        img = Image.open(image_path)

        img.thumbnail((350, 350)) 
        photo = ImageTk.PhotoImage(img)
        
        image_display_label_gui.config(image=photo)
        image_display_label_gui.image = photo 
        
    except Exception as e:
        if caption_display_label_gui:
            caption_display_label_gui.config(text=f"Error displaying image: {e}", foreground="red")
        if image_display_label_gui: 
            image_display_label_gui.config(image=None)
            image_display_label_gui.image = None


def generate_caption_for_selected_image_gui():
    """Generates and displays the caption for the selected image in the GUI."""
    global selected_image_path_gui, caption_display_label_gui, progress_bar_gui, root_gui
    
    if selected_image_path_gui:
        caption_display_label_gui.config(text="Generating caption, please wait...", foreground="blue")
        if progress_bar_gui:
            progress_bar_gui.start(10) 
        if root_gui:
            root_gui.update_idletasks() 

        try:
            
            generated_caption = greedy_algorithm(selected_image_path_gui)
            caption_display_label_gui.config(text=f"{generated_caption}", wraplength=380, foreground="black")
        except NameError as ne:
            
            error_text = (f"NameError: {ne}. A required function or variable for caption generation "
                          f"(like 'greedy_algorithm', 'mm1', 'new_v', 'INDEX_TO_WORD', etc.) "
                          f"is not defined. Please ensure all prerequisite notebook cells are run "
                          f"successfully before starting the GUI.")
            caption_display_label_gui.config(text=error_text, foreground="red", wraplength=380)
        except Exception as e:
            caption_display_label_gui.config(text=f"Error during caption generation: {e}", foreground="red")
        finally:
            if progress_bar_gui:
                progress_bar_gui.stop() 
    else:
        caption_display_label_gui.config(text="Please select an image first.", foreground="black")

def select_image_action_gui():
    """Handles the image selection button action."""
    global selected_image_path_gui, caption_display_label_gui, image_display_label_gui
    

    if image_display_label_gui:

        placeholder_img = Image.new('RGBA', (350,250), (0,0,0,0)) 
        placeholder_photo = ImageTk.PhotoImage(placeholder_img)
        image_display_label_gui.config(image=placeholder_photo)
        image_display_label_gui.image = placeholder_photo
        
    caption_display_label_gui.config(text="Select an image to generate a caption.", foreground="black")

    file_path = filedialog.askopenfilename(
        title="Select an Image File",
        filetypes=(("JPEG files", "*.jpg;*.jpeg"),
                   ("PNG files", "*.png"),
                   ("All files", "*.*"))
    )
    if file_path:
        selected_image_path_gui = file_path
        display_image_gui(selected_image_path_gui)
        generate_caption_for_selected_image_gui() 
        
    else:
        selected_image_path_gui = None
        caption_display_label_gui.config(text="No image selected.", foreground="black")


def create_image_caption_gui():
    """Creates and runs the main Tkinter GUI."""
    global image_display_label_gui, caption_display_label_gui, root_gui, progress_bar_gui


    if tk._default_root:
        print("Attempting to close existing Tkinter root window...")
        try:
            tk._default_root.destroy()
            tk._default_root = None
        except tk.TclError:
            print("Could not destroy previous Tk root, a new one might not work as expected.")


    root_gui = tk.Tk()
    root_gui.title("Image Caption Generator")
    root_gui.geometry("420x650") 

    style = ttk.Style()
    style.theme_use('clam') 
    style.configure("TButton", font=("Helvetica", 12), padding=10)
    style.configure("TLabel", font=("Helvetica", 11)) 
    style.configure("Header.TLabel", font=("Helvetica", 14, "bold"))
    style.configure("Caption.TLabel", font=("Helvetica", 12), padding=5)



    main_frame = ttk.Frame(root_gui, padding="15 15 15 15")
    main_frame.pack(expand=True, fill=tk.BOTH)


    select_button = ttk.Button(main_frame, text="📷 Select Image", command=select_image_action_gui)
    select_button.pack(pady=(0, 15), fill=tk.X)

   
   
    image_frame = ttk.Frame(main_frame, relief="sunken", borderwidth=2, width=350, height=250)
    image_frame.pack(pady=10, fill=tk.BOTH, expand=False) 
    image_frame.pack_propagate(False) 
    

    image_display_label_gui = ttk.Label(image_frame, anchor="center")
    image_display_label_gui.pack(expand=True, fill=tk.BOTH)
    

    placeholder_img = Image.new('RGBA', (350,250), (230,230,230,255)) 

    placeholder_photo = ImageTk.PhotoImage(placeholder_img)
    image_display_label_gui.config(image=placeholder_photo)
    image_display_label_gui.image = placeholder_photo
    


    progress_bar_gui = ttk.Progressbar(main_frame, mode='indeterminate')
    progress_bar_gui.pack(pady=10, fill='x')

    caption_header_label = ttk.Label(main_frame, text="✒️ Generated Caption:", style="Header.TLabel")
    caption_header_label.pack(pady=(10,5))
    

    caption_display_frame = ttk.Frame(main_frame, relief="groove", borderwidth=1)
    caption_display_frame.pack(pady=5, expand=True, fill="both")

    caption_display_label_gui = ttk.Label(
        caption_display_frame,
        text="Select an image to see the generated caption here.",
        wraplength=370, 
        justify="center",
        anchor="center",
        style="Caption.TLabel"
    )
    caption_display_label_gui.pack(expand=True, fill="both", padx=5, pady=5)



    missing_components = []
    try: _ = mm1
    except NameError: missing_components.append("model (mm1)")
    try: _ = new_v
    except NameError: missing_components.append("TextVectorization layer (new_v)")
    try: _ = INDEX_TO_WORD
    except NameError: missing_components.append("INDEX_TO_WORD mapping")
    try: _ = MAX_DECODED_SENTENCE_LENGTH
    except NameError: missing_components.append("MAX_DECODED_SENTENCE_LENGTH")
    try: _ = decode_and_resize
    except NameError: missing_components.append("decode_and_resize function")
    try: _ = greedy_algorithm
    except NameError: missing_components.append("greedy_algorithm function")

    if missing_components:
        error_msg = ("Error: Critical components are not loaded/defined: " + 
                     ", ".join(missing_components) + 
                     ".\nPlease ensure all prerequisite notebook cells have been run successfully.")
        caption_display_label_gui.config(text=error_msg, foreground="red", wraplength=380)
        select_button.config(state=tk.DISABLED) 

    root_gui.mainloop()

In [ ]:
create_image_caption_gui()

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk 
from PIL import Image, ImageTk
import tensorflow as tf
import keras 
from keras import layers
from keras.applications import efficientnet 
from tensorflow.keras.utils import to_categorical, plot_model 
from keras.layers import TextVectorization 

import numpy as np
import pickle
import re
import os
import threading

IMAGE_SIZE = (299, 299)
SEQ_LENGTH = 25
VOCAB_SIZE = 10000
EMBED_DIM = 512
FF_DIM = 512

MODEL_WEIGHTS_PATH = "model_weights.h5"
VECTORIZER_PATH = "tv_layer.pkl"

vectorizer_global = None
caption_model_global = None
index_to_word_global = None
uploaded_image_path_global = None
uploaded_image_bytes_global = None


def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    strip_chars = "!\\\"#$%&'()*+,-./:;=?@[\\]^_`{|}~1234567890"
    return tf.strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")

def decode_and_resize(img_input, is_path=True):
    if is_path:
        img = tf.io.read_file(img_input)
    else: 
        img = img_input
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img

class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.0)
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.dense_1 = layers.Dense(embed_dim, activation="relu")

    def call(self, inputs, training, mask=None):
        inputs = self.layernorm_1(inputs)
        inputs = self.dense_1(inputs)
        attention_output_1 = self.attention_1(query=inputs,
                                              value=inputs,
                                              key=inputs,
                                              attention_mask=None,
                                              training=training)
        out_1 = self.layernorm_2(inputs + attention_output_1)
        return out_1

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.embed_scale = tf.math.sqrt(tf.cast(embed_dim, tf.float32))

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_tokens = embedded_tokens * self.embed_scale
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return tf.math.not_equal(inputs, 0)

class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, ff_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.cross_attention_2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.ffn_layer_1 = layers.Dense(ff_dim, activation="relu")
        self.ffn_layer_2 = layers.Dense(embed_dim)

        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()

        self.embedding = PositionalEmbedding(embed_dim=EMBED_DIM,
                                             sequence_length=SEQ_LENGTH,
                                             vocab_size=VOCAB_SIZE)
        self.out = layers.Dense(VOCAB_SIZE, activation="softmax")

        self.dropout_1 = layers.Dropout(0.3)
        self.dropout_2 = layers.Dropout(0.5)
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, training, mask=None):
        inputs = self.embedding(inputs)
        causal_mask = self.get_causal_attention_mask(inputs)
        
        if mask is not None:
            padding_mask_mha1 = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32)
            combined_mask = tf.minimum(padding_mask_mha1, causal_mask)
            padding_mask_cross = tf.cast(mask[:, :, tf.newaxis], dtype=tf.int32)
        else:
            combined_mask = causal_mask
            padding_mask_cross = None

        attention_output_1 = self.attention_1(query=inputs,
                                              value=inputs,
                                              key=inputs,
                                              attention_mask=combined_mask,
                                              training=training)
        out_1 = self.layernorm_1(inputs + attention_output_1)
        
        cross_attention_output_2 = self.cross_attention_2(query=out_1,
                                              value=encoder_outputs,
                                              key=encoder_outputs,
                                              attention_mask=padding_mask_cross,
                                              training=training)
        out_2 = self.layernorm_2(out_1 + cross_attention_output_2)

        ffn_out = self.ffn_layer_1(out_2)
        ffn_out = self.dropout_1(ffn_out, training=training)
        ffn_out = self.ffn_layer_2(ffn_out)

        ffn_out = self.layernorm_3(ffn_out + out_2, training=training)
        ffn_out = self.dropout_2(ffn_out, training=training)
        
        preds = self.out(ffn_out)
        return preds
    
    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, sequence_length, sequence_length))
        mult = tf.concat([tf.expand_dims(batch_size, -1),tf.constant([1, 1], dtype=tf.int32)],axis=0)
        return tf.tile(mask, mult)

def get_cnn_model():
    base_model = efficientnet.EfficientNetB0(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights="imagenet")
    base_model.trainable = False
    base_model_out = base_model.output
    base_model_out = layers.Reshape((-1, base_model_out.shape[-1]))(base_model_out)
    cnn_model = keras.models.Model(base_model.input, base_model_out)
    return cnn_model

class ImageCaptioningModel(keras.Model):
    def __init__(self, cnn_model, encoder, decoder, num_captions_per_image=5, image_aug=None):
        super().__init__()
        self.cnn_model = cnn_model
        self.encoder = encoder
        self.decoder = decoder
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.acc_tracker = keras.metrics.Mean(name="accuracy")
        self.num_captions_per_image = num_captions_per_image
        self.image_aug = image_aug

    def call(self, inputs, training=False):
        image_input, caption_input = inputs
        img_features = self.cnn_model(image_input, training=training)
        encoded_img = self.encoder(img_features, training=training)
        mask = tf.math.not_equal(caption_input, 0)
        predictions = self.decoder(caption_input, encoded_img, training=training, mask=mask)
        return predictions

class LRSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, post_warmup_learning_rate, warmup_steps):
        super().__init__()
        self.post_warmup_learning_rate = post_warmup_learning_rate
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        global_step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        warmup_progress = global_step / warmup_steps
        warmup_learning_rate = self.post_warmup_learning_rate * warmup_progress
        return tf.cond(
            global_step < warmup_steps,
            lambda: warmup_learning_rate,
            lambda: self.post_warmup_learning_rate)


def _load_models_task():
    global vectorizer_global, caption_model_global, index_to_word_global, root_app

    try:
        root_app.update_status("Loading Text Vectorizer...")
        with open(VECTORIZER_PATH, "rb") as f:
            from_disk = pickle.load(f)
        cfg, wts = from_disk["config"], from_disk["weights"]
        vocab_list_bytes = list(wts[0])
        
        if vocab_list_bytes and isinstance(vocab_list_bytes[0], bytes):
            vocab_list = [v.decode('utf-8') for v in vocab_list_bytes]
        else:
            vocab_list = vocab_list_bytes

        vectorizer_global = TextVectorization(
            max_tokens=cfg["max_tokens"],
            output_mode=cfg["output_mode"],
            ngrams=cfg.get("ngrams"),
            standardize=custom_standardization,
            split=cfg["split"],
            output_sequence_length=cfg["output_sequence_length"],
            pad_to_max_tokens=cfg.get("pad_to_max_tokens", False),
            vocabulary=vocab_list
        )
        vocab = vectorizer_global.get_vocabulary()
        index_to_word_global = {idx: word for idx, word in enumerate(vocab)}
        root_app.update_status("Text Vectorizer loaded.")
    except Exception as e:
        root_app.update_status(f"Error loading vectorizer: {e}", "red")
        messagebox.showerror("Error", f"Failed to load Text Vectorizer: {e}")
        return

    try:
        root_app.update_status("Loading Captioning Model...")
        cnn_model_instance = get_cnn_model()
        encoder_instance = TransformerEncoderBlock(embed_dim=EMBED_DIM, dense_dim=FF_DIM, num_heads=2)
        decoder_instance = TransformerDecoderBlock(embed_dim=EMBED_DIM, ff_dim=FF_DIM, num_heads=3)
        
        caption_model_instance = ImageCaptioningModel(
            cnn_model=cnn_model_instance, 
            encoder=encoder_instance, 
            decoder=decoder_instance,
            image_aug=None 
        )
        
        num_train_steps_placeholder = 3000 
        num_warmup_steps_placeholder = num_train_steps_placeholder // 15
        lr_schedule_instance = LRSchedule(post_warmup_learning_rate=1e-4, warmup_steps=num_warmup_steps_placeholder)
        cross_entropy_loss = keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')
        caption_model_instance.compile(optimizer=keras.optimizers.Adam(lr_schedule_instance), loss=cross_entropy_loss)
        
        dummy_image = tf.zeros((1, *IMAGE_SIZE, 3))
        dummy_caption_input = tf.zeros((1, SEQ_LENGTH -1), dtype=tf.int32) 
        img_features_dummy = caption_model_instance.cnn_model(dummy_image, training=False)
        encoded_dummy = caption_model_instance.encoder(img_features_dummy, training=False)
        mask_dummy = tf.cast(tf.math.not_equal(dummy_caption_input, 0), dtype=tf.bool)
        _ = caption_model_instance.decoder(dummy_caption_input, encoded_dummy, training=False, mask=mask_dummy)
        root_app.update_status("Model compiled and built.")

        caption_model_instance.load_weights(MODEL_WEIGHTS_PATH)
        caption_model_global = caption_model_instance
        root_app.update_status("Captioning Model loaded successfully.", "green")
        root_app.enable_upload_button()

    except Exception as e:
        root_app.update_status(f"Error loading model: {e}", "red")
        messagebox.showerror("Error", f"Failed to load Captioning Model: {e}")


def load_models_threaded():
    if root_app.load_model_button:
        root_app.load_model_button.config(state=tk.DISABLED)
    thread = threading.Thread(target=_load_models_task)
    thread.daemon = True
    thread.start()

def generate_caption_greedy_tk(image_bytes_for_tf):
    global caption_model_global, vectorizer_global, index_to_word_global
    
    if not all([caption_model_global, vectorizer_global, index_to_word_global]):
        messagebox.showerror("Error", "Model or vectorizer not loaded.")
        return "Error: Model not loaded."

    MAX_DECODED_SENTENCE_LENGTH = SEQ_LENGTH - 1
    img_tensor = decode_and_resize(image_bytes_for_tf, is_path=False)
    img_tensor_expanded = tf.expand_dims(img_tensor, 0)

    img_features = caption_model_global.cnn_model(img_tensor_expanded, training=False)
    encoded_img = caption_model_global.encoder(img_features, training=False)

    decoded_caption = "<start> "
    for i in range(MAX_DECODED_SENTENCE_LENGTH):
        tokenized_caption_full = vectorizer_global([decoded_caption])
        tokenized_caption_input = tokenized_caption_full[:, :-1]
        mask = tf.math.not_equal(tokenized_caption_input, 0)
        predictions = caption_model_global.decoder(tokenized_caption_input, encoded_img, training=False, mask=mask)
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = index_to_word_global.get(sampled_token_index, "<unk>")
        if sampled_token == "<end>":
            break
        decoded_caption += sampled_token + " "
    
    final_caption = decoded_caption.replace("<start> ", "").replace(" <end>", "").strip()
    return final_caption

class App:
    def __init__(self, root):
        self.root = root
        root.title("🖼️ Image Caption Generator")
        root.geometry("950x750") 
        root.configure(bg="#e0e0e0")

        self.style = ttk.Style()
        self.style.theme_use('clam') 

        self.style.configure("TFrame", background="#e0e0e0")
        self.style.configure("Header.TLabel", font=("Helvetica", 20, "bold"), foreground="#333", background="#e0e0e0")
        self.style.configure("Status.TLabel", font=("Helvetica", 11), background="#e0e0e0")
        self.style.configure("Footer.TLabel", font=("Helvetica", 10, "italic"), foreground="#555", background="#d0d0d0")
        self.style.configure("TButton", font=("Helvetica", 13, "bold"), padding=10, relief="raised")
        self.style.map("TButton",
            foreground=[('pressed', 'red'), ('active', '#3498db')],
            background=[('pressed', '!disabled', 'black'), ('active', 'white')]
        )
        self.style.configure("ImageFrame.TFrame", background="#ffffff", relief="sunken", borderwidth=2)
        self.style.configure("Caption.TLabel", font=("Georgia", 16), foreground="#2c3e50", background="#f0f0f0", padding=10)

        app_frame = ttk.Frame(root, padding="20 20 20 20")
        app_frame.pack(fill=tk.BOTH, expand=True)

        header_label = ttk.Label(app_frame, text="✨ Image to Caption ✨", style="Header.TLabel")
        header_label.pack(pady=(0, 25))

        control_frame = ttk.Frame(app_frame, padding="10 0 10 0")
        control_frame.pack(fill=tk.X, pady=10)
        control_frame.columnconfigure(0, weight=1) 
        control_frame.columnconfigure(1, weight=1)

        self.load_model_button = ttk.Button(control_frame, text="🔌 Load Models & Vectorizer", command=load_models_threaded)
        self.load_model_button.grid(row=0, column=0, padx=15, sticky="ew")

        self.upload_button = ttk.Button(control_frame, text="🖼️ Upload Image", command=self.upload_image_action, state=tk.DISABLED)
        self.upload_button.grid(row=0, column=1, padx=15, sticky="ew")
        
        self.status_label = ttk.Label(app_frame, text="Welcome! Please load models to begin.", style="Status.TLabel", foreground="navy")
        self.status_label.pack(pady=10)

        display_frame = ttk.Frame(app_frame, padding="10 10 10 10")
        display_frame.pack(fill=tk.BOTH, expand=True, pady=15)
        
        display_frame.columnconfigure(0, weight=1) 
        display_frame.columnconfigure(1, weight=1) 
        display_frame.rowconfigure(0, weight=1)    

        self.image_frame = ttk.Frame(display_frame, style="ImageFrame.TFrame", width=420, height=420) 
        self.image_frame.grid(row=0, column=0, padx=15, pady=10, sticky="nsew")
        self.image_frame.pack_propagate(False) 

        self.image_label = ttk.Label(self.image_frame, text="Your Image Here", anchor="center", background="#ffffff")
        self.image_label.pack(fill=tk.BOTH, expand=True)
        self.update_image_placeholder()



        caption_area_frame = ttk.Frame(display_frame, padding=5)
        caption_area_frame.grid(row=0, column=1, padx=15, pady=10, sticky="nsew")
        caption_area_frame.rowconfigure(0, weight=0) 
        caption_area_frame.rowconfigure(1, weight=1) 
        caption_area_frame.columnconfigure(0, weight=1)

        caption_title = ttk.Label(caption_area_frame, text="Generated Caption:", font=("Helvetica", 16, "bold"), background="#e0e0e0")
        caption_title.grid(row=0, column=0, pady=(0,10), sticky="w")

        self.caption_text_widget = tk.Text(
            caption_area_frame, 
            wrap=tk.WORD, 
            relief=tk.FLAT, 
            borderwidth=2, 
            font=("Georgia", 17), 
            padx=10, pady=10,
            state=tk.DISABLED,
            bg="#f9f9f9", 
            selectbackground="#add8e6" 
        )
        self.caption_text_widget.grid(row=1, column=0, sticky="nsew")

        footer_frame = ttk.Frame(root, padding="10 10 10 10", style="Footer.TFrame") 
        footer_frame.pack(side=tk.BOTTOM, fill=tk.X)
        
        name_label = ttk.Label(footer_frame, text="Developed by Mayank Gour", style="Footer.TLabel", anchor="center")
        name_label.pack(expand=True)


        if not os.path.exists(MODEL_WEIGHTS_PATH) or not os.path.exists(VECTORIZER_PATH):
            self.update_status("Error: Model or vectorizer file not found. Check paths.", "red")
            self.load_model_button.config(state=tk.DISABLED)


    def update_image_placeholder(self):

        try:
            placeholder = Image.new('RGB', (400, 400), color = '#f0f0f0')
          
           
            photo = ImageTk.PhotoImage(placeholder)
            self.image_label.config(image=photo)
            self.image_label.image = photo
        except Exception as e:
            print(f"Error creating placeholder: {e}")
            self.image_label.config(text="Image Preview Area")


    def update_status(self, message, color="navy"): # Default color navy
        if hasattr(self, 'status_label') and self.status_label.winfo_exists():
            self.root.after(0, lambda: self.status_label.config(text=message, foreground=color))
        print(f"Status: {message}")

    def enable_upload_button(self):
         if hasattr(self, 'upload_button') and self.upload_button.winfo_exists():
            self.root.after(0, lambda: self.upload_button.config(state=tk.NORMAL))

    def display_image(self, file_path):
        global uploaded_image_bytes_global
        try:
            img_pil = Image.open(file_path)
            
            with open(file_path, 'rb') as f:
                uploaded_image_bytes_global = f.read()

            # Resize for display in Tkinter label, fitting within the image_frame
            target_width = self.image_frame.winfo_width() if self.image_frame.winfo_width() > 10 else 400
            target_height = self.image_frame.winfo_height() if self.image_frame.winfo_height() > 10 else 400
            
            img_pil.thumbnail((target_width - 10, target_height - 10)) # -10 for some padding
            
            photo = ImageTk.PhotoImage(img_pil)
            self.image_label.config(image=photo, text="")
            self.image_label.image = photo 
        except Exception as e:
            messagebox.showerror("Image Display Error", f"Could not display image: {e}")
            self.image_label.config(text="Error displaying image", image=None)
            self.update_image_placeholder()


    def upload_image_action(self):
        global uploaded_image_path_global, uploaded_image_bytes_global
        
        if not caption_model_global or not vectorizer_global:
            messagebox.showwarning("Models Not Loaded", "Please load the models before uploading an image.")
            return

        file_path = filedialog.askopenfilename(
            title="Select an Image File",
            filetypes=(("JPEG files", "*.jpg;*.jpeg"), ("PNG files", "*.png"), ("All files", "*.*"))
        )
        if file_path:
            uploaded_image_path_global = file_path
            self.update_status(f"Image selected: {os.path.basename(file_path)}", "darkgreen")
            self.display_image(file_path)
            
            self.caption_text_widget.config(state=tk.NORMAL)
            self.caption_text_widget.delete(1.0, tk.END)
            self.caption_text_widget.config(state=tk.DISABLED)
            
            self.generate_and_display_caption()
        else:
            self.update_status("Image selection cancelled.", "orange")
            uploaded_image_bytes_global = None

    def generate_and_display_caption(self):
        global uploaded_image_bytes_global
        if uploaded_image_bytes_global:
            self.update_status("Generating caption...", "blue")
            self.caption_text_widget.config(state=tk.NORMAL)
            self.caption_text_widget.delete(1.0, tk.END)
            self.caption_text_widget.insert(tk.END, "Thinking... ")
            self.caption_text_widget.config(state=tk.DISABLED)
            self.root.update_idletasks()

            try:
                def _task():
                    caption = generate_caption_greedy_tk(uploaded_image_bytes_global)
                    self.root.after(0, self._update_caption_widget, caption)
                threading.Thread(target=_task, daemon=True).start()
            except Exception as e:
                messagebox.showerror("Caption Generation Error", f"Failed to generate caption: {e}")
                self._update_caption_widget(f"Error during generation: {e}")
        else:
            self.update_status("No image selected for captioning.", "orange")
            
    def _update_caption_widget(self, caption_text):
        self.caption_text_widget.config(state=tk.NORMAL)
        self.caption_text_widget.delete(1.0, tk.END)
        self.caption_text_widget.insert(tk.END, caption_text)
        self.caption_text_widget.config(state=tk.DISABLED)
        if "Error" not in caption_text:
            self.update_status("Caption generated successfully!", "green")
        else:
            self.update_status("Error in caption generation.", "red")


if __name__ == '__main__':
    root = tk.Tk()
    root_app = App(root) 
    root.mainloop()

In [ ]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk   

class CaptionApp(ttk.Frame):
    PREVIEW_SIZE = 480            

    def __init__(self, master):
        super().__init__(master)
        master.title("Image Captioning")
        master.geometry("960x720")           
        master.minsize(800, 600)
        master.configure(bg="#1e1e1e")       
        self.pack(fill="both", expand=True)

        style = ttk.Style(master)
        style.theme_use("clam")              
        style.configure(".",           background="#1e1e1e", foreground="#ffffff")
        style.configure("Header.TLabel",     font=("Segoe UI", 22, "bold"))
        style.configure("Caption.TEntry",    font=("Segoe UI", 12))
        style.configure("TButton",           font=("Segoe UI", 11), padding=6)
        style.map("TButton", foreground=[("disabled", "#888")])

        self.columnconfigure(0, weight=1)
        self.rowconfigure(2, weight=1)

        ttk.Label(self, text="Image-to-Caption", style="Header.TLabel"
                 ).grid(row=0, column=0, pady=(20, 10))

        self.preview = tk.Label(self, bg="#2a2a2a", width=self.PREVIEW_SIZE,
                                height=self.PREVIEW_SIZE)
        self.preview.grid(row=1, column=0, pady=10)

        self.caption_var = tk.StringVar()
        cap_frame = ttk.Frame(self)
        cap_frame.grid(row=2, column=0, sticky="nsew", padx=30, pady=10)
        cap_frame.columnconfigure(0, weight=1)
        ttk.Label(cap_frame, text="Caption:").grid(row=0, column=0, sticky="w")
        ttk.Entry(cap_frame, textvariable=self.caption_var, state="readonly",
                  style="Caption.TEntry").grid(row=1, column=0, sticky="ew", pady=4)

        btn_frame = ttk.Frame(self)
        btn_frame.grid(row=3, column=0, pady=10)
        ttk.Button(btn_frame, text="Open image", command=self.open_image
                  ).pack(side="left", padx=8)
        ttk.Button(btn_frame, text="Generate caption", command=self.gen_caption
                  ).pack(side="left", padx=8)

        ttk.Label(self, text="Made with by Mayank Gour", font=("Segoe UI", 10)
                 ).grid(row=4, column=0, pady=(20, 10))

        self.path   = None
        self.tk_img = None          

    def open_image(self):
        ft = [("Images", "*.jpg *.jpeg *.png *.bmp *.gif"), ("All files", "*.*")]
        path = filedialog.askopenfilename(title="Choose an image", filetypes=ft)
        if not path:
            return

        self.path = path
        try:
            thumb = Image.open(path).convert("RGB")
            thumb.thumbnail((self.PREVIEW_SIZE, self.PREVIEW_SIZE))
            # center on a square canvas
            canvas = Image.new("RGB", (self.PREVIEW_SIZE, self.PREVIEW_SIZE),
                               color="#2a2a2a")
            x = (self.PREVIEW_SIZE - thumb.width) // 2
            y = (self.PREVIEW_SIZE - thumb.height) // 2
            canvas.paste(thumb, (x, y))
            self.tk_img = ImageTk.PhotoImage(canvas)
            self.preview.configure(image=self.tk_img)
            self.caption_var.set("")          
        except Exception as e:
            messagebox.showerror("Error", str(e))
            self.path = None

    def gen_caption(self):
        if not self.path:
            messagebox.showinfo("Info", "Please select an image first.")
            return
        try:
            caption = greedy_algorithm(self.path)   
            self.caption_var.set(caption)
        except Exception as e:
            messagebox.showerror("Error", str(e))

if __name__ == "__main__":
    root = tk.Tk()
    CaptionApp(root)
    root.mainloop()



In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from PIL import Image, ImageTk, ImageDraw, ImageFont 
import tensorflow as tf
import keras
from keras import layers
from keras.applications import efficientnet
from tensorflow.keras.utils import to_categorical, plot_model
from keras.layers import TextVectorization

import numpy as np
import pickle
import re
import os
import threading

IMAGE_SIZE = (299, 299)
SEQ_LENGTH = 25
VOCAB_SIZE = 10000
EMBED_DIM = 512
FF_DIM = 512

MODEL_WEIGHTS_PATH = "model_weights.h5"
VECTORIZER_PATH = "tv_layer.pkl"

vectorizer_global = None
caption_model_global = None
index_to_word_global = None
uploaded_image_path_global = None
uploaded_image_bytes_global = None


def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    strip_chars = "!\\\"#$%&'()*+,-./:;=?@[\\]^_`{|}~1234567890"
    return tf.strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")

def decode_and_resize(img_input, is_path=True):
    if is_path:
        img = tf.io.read_file(img_input)
    else: 
        img = img_input
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img

class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.0)
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.dense_1 = layers.Dense(embed_dim, activation="relu")

    def call(self, inputs, training, mask=None):
        inputs = self.layernorm_1(inputs)
        inputs = self.dense_1(inputs)
        attention_output_1 = self.attention_1(query=inputs,
                                              value=inputs,
                                              key=inputs,
                                              attention_mask=None,
                                              training=training)
        out_1 = self.layernorm_2(inputs + attention_output_1)
        return out_1

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.embed_scale = tf.math.sqrt(tf.cast(embed_dim, tf.float32))

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_tokens = embedded_tokens * self.embed_scale
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return tf.math.not_equal(inputs, 0)

class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, ff_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.cross_attention_2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.ffn_layer_1 = layers.Dense(ff_dim, activation="relu")
        self.ffn_layer_2 = layers.Dense(embed_dim)

        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()

        self.embedding = PositionalEmbedding(embed_dim=EMBED_DIM,
                                             sequence_length=SEQ_LENGTH,
                                             vocab_size=VOCAB_SIZE)
        self.out = layers.Dense(VOCAB_SIZE, activation="softmax")

        self.dropout_1 = layers.Dropout(0.3)
        self.dropout_2 = layers.Dropout(0.5)
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, training, mask=None):
        inputs = self.embedding(inputs)
        causal_mask = self.get_causal_attention_mask(inputs)
        
        if mask is not None:
            padding_mask_mha1 = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32)
            combined_mask = tf.minimum(padding_mask_mha1, causal_mask)
            padding_mask_cross = tf.cast(mask[:, :, tf.newaxis], dtype=tf.int32)
        else:
            combined_mask = causal_mask
            padding_mask_cross = None

        attention_output_1 = self.attention_1(query=inputs,
                                              value=inputs,
                                              key=inputs,
                                              attention_mask=combined_mask,
                                              training=training)
        out_1 = self.layernorm_1(inputs + attention_output_1)
        
        cross_attention_output_2 = self.cross_attention_2(query=out_1,
                                              value=encoder_outputs,
                                              key=encoder_outputs,
                                              attention_mask=padding_mask_cross,
                                              training=training)
        out_2 = self.layernorm_2(out_1 + cross_attention_output_2)

        ffn_out = self.ffn_layer_1(out_2)
        ffn_out = self.dropout_1(ffn_out, training=training)
        ffn_out = self.ffn_layer_2(ffn_out)

        ffn_out = self.layernorm_3(ffn_out + out_2, training=training)
        ffn_out = self.dropout_2(ffn_out, training=training)
        
        preds = self.out(ffn_out)
        return preds
    
    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, sequence_length, sequence_length))
        mult = tf.concat([tf.expand_dims(batch_size, -1),tf.constant([1, 1], dtype=tf.int32)],axis=0)
        return tf.tile(mask, mult)

def get_cnn_model():
    base_model = efficientnet.EfficientNetB0(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights="imagenet")
    base_model.trainable = False
    base_model_out = base_model.output
    base_model_out = layers.Reshape((-1, base_model_out.shape[-1]))(base_model_out)
    cnn_model = keras.models.Model(base_model.input, base_model_out)
    return cnn_model

class ImageCaptioningModel(keras.Model):
    def __init__(self, cnn_model, encoder, decoder, num_captions_per_image=5, image_aug=None):
        super().__init__()
        self.cnn_model = cnn_model
        self.encoder = encoder
        self.decoder = decoder
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.acc_tracker = keras.metrics.Mean(name="accuracy")
        self.num_captions_per_image = num_captions_per_image
        self.image_aug = image_aug

    def call(self, inputs, training=False):
        image_input, caption_input = inputs
        img_features = self.cnn_model(image_input, training=training)
        encoded_img = self.encoder(img_features, training=training)
        mask = tf.math.not_equal(caption_input, 0)
        predictions = self.decoder(caption_input, encoded_img, training=training, mask=mask)
        return predictions

class LRSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, post_warmup_learning_rate, warmup_steps):
        super().__init__()
        self.post_warmup_learning_rate = post_warmup_learning_rate
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        global_step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        warmup_progress = global_step / warmup_steps
        warmup_learning_rate = self.post_warmup_learning_rate * warmup_progress
        return tf.cond(
            global_step < warmup_steps,
            lambda: warmup_learning_rate,
            lambda: self.post_warmup_learning_rate)


def _load_models_task(): 
    global vectorizer_global, caption_model_global, index_to_word_global, root_app
    try:
        root_app.update_status("Loading Text Vectorizer...")
        with open(VECTORIZER_PATH, "rb") as f: from_disk = pickle.load(f)
        cfg, wts = from_disk["config"], from_disk["weights"]
        vocab_list_bytes = list(wts[0])
        vocab_list = [v.decode('utf-8') for v in vocab_list_bytes if isinstance(v, bytes)] if vocab_list_bytes and isinstance(vocab_list_bytes[0], bytes) else vocab_list_bytes
        vectorizer_global = TextVectorization(max_tokens=cfg["max_tokens"], output_mode=cfg["output_mode"], ngrams=cfg.get("ngrams"), standardize=custom_standardization, split=cfg["split"], output_sequence_length=cfg["output_sequence_length"], pad_to_max_tokens=cfg.get("pad_to_max_tokens", False), vocabulary=vocab_list)
        index_to_word_global = {idx: word for idx, word in enumerate(vectorizer_global.get_vocabulary())}
        root_app.update_status("Text Vectorizer loaded.")
    except Exception as e:
        root_app.update_status(f"Error loading vectorizer: {str(e)[:100]}", "red")
        messagebox.showerror("Error", f"Failed to load Text Vectorizer: {e}")
        return
    try:
        root_app.update_status("Loading Captioning Model...")
        cnn_model_instance = get_cnn_model()
        encoder_instance = TransformerEncoderBlock(embed_dim=EMBED_DIM, dense_dim=FF_DIM, num_heads=2)
        decoder_instance = TransformerDecoderBlock(embed_dim=EMBED_DIM, ff_dim=FF_DIM, num_heads=3)
        caption_model_instance = ImageCaptioningModel(cnn_model=cnn_model_instance, encoder=encoder_instance, decoder=decoder_instance)
        lr_schedule_instance = LRSchedule(post_warmup_learning_rate=1e-4, warmup_steps=(3000 // 15))
        caption_model_instance.compile(optimizer=keras.optimizers.Adam(lr_schedule_instance), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none'))
        dummy_image = tf.zeros((1, *IMAGE_SIZE, 3)); dummy_caption_input = tf.zeros((1, SEQ_LENGTH -1), dtype=tf.int32)
        img_features_dummy = caption_model_instance.cnn_model(dummy_image, training=False); encoded_dummy = caption_model_instance.encoder(img_features_dummy, training=False)
        _ = caption_model_instance.decoder(dummy_caption_input, encoded_dummy, training=False, mask=tf.cast(tf.math.not_equal(dummy_caption_input, 0), dtype=tf.bool))
        root_app.update_status("Model compiled & built.")
        caption_model_instance.load_weights(MODEL_WEIGHTS_PATH)
        caption_model_global = caption_model_instance
        root_app.update_status("Captioning Model loaded successfully!", "green")
        root_app.enable_upload_button()
    except Exception as e:
        root_app.update_status(f"Error loading model: {str(e)[:100]}", "red")
        messagebox.showerror("Error", f"Failed to load Captioning Model: {e}")

def load_models_threaded(): # Same as before
    if root_app.load_model_button: root_app.load_model_button.config(state=tk.DISABLED)
    threading.Thread(target=_load_models_task, daemon=True).start()

# --- Inference Function (remains the same) ---
def generate_caption_greedy_tk(image_bytes_for_tf): # Same as before
    global caption_model_global, vectorizer_global, index_to_word_global
    if not all([caption_model_global, vectorizer_global, index_to_word_global]): return "Error: Model not loaded."
    MAX_DECODED_SENTENCE_LENGTH = SEQ_LENGTH - 1
    img_tensor = decode_and_resize(image_bytes_for_tf, is_path=False)
    img_tensor_expanded = tf.expand_dims(img_tensor, 0)
    img_features = caption_model_global.cnn_model(img_tensor_expanded, training=False)
    encoded_img = caption_model_global.encoder(img_features, training=False)
    decoded_caption = "<start> "
    for i in range(MAX_DECODED_SENTENCE_LENGTH):
        tokenized_caption_full = vectorizer_global([decoded_caption])
        tokenized_caption_input = tokenized_caption_full[:, :-1]
        mask = tf.math.not_equal(tokenized_caption_input, 0)
        predictions = caption_model_global.decoder(tokenized_caption_input, encoded_img, training=False, mask=mask)
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = index_to_word_global.get(sampled_token_index, "<unk>")
        if sampled_token == "<end>": break
        decoded_caption += sampled_token + " "
    return decoded_caption.replace("<start> ", "").replace(" <end>", "").strip()

class App:
    def __init__(self, root):
        self.root = root
        root.title("Vision Scribe: Image to Text by Mayank Gour")
        root.geometry("1000x800") 
        self.app_bg_color = "#f0f2f5" 
        self.card_bg_color = "#ffffff" 
        self.text_color = "#333333"   
        self.primary_color = "#007bff" 
        self.secondary_color = "#6c757d" 
        self.font_family = "Segoe UI" 

        root.configure(bg=self.app_bg_color)

        self.style = ttk.Style()
        try:
            current_os = root.tk.call('tk', 'windowingsystem')
            if current_os == 'win32': self.style.theme_use('vista')
            elif current_os == 'aqua': self.style.theme_use('aqua')
            else: self.style.theme_use('clam')
        except tk.TclError:
            self.style.theme_use('clam')


        self.style.configure("TFrame", background=self.app_bg_color)
        self.style.configure("Card.TFrame", background=self.card_bg_color, relief="flat", borderwidth=1) 
        
        self.style.configure("Header.TLabel", font=(self.font_family, 26, "bold"), foreground="#2c3e50", background=self.app_bg_color)
        self.style.configure("SubHeader.TLabel", font=(self.font_family, 16, "normal"), foreground=self.secondary_color, background=self.app_bg_color)
        self.style.configure("Status.TLabel", font=(self.font_family, 12), background=self.app_bg_color)
        self.style.configure("Footer.TLabel", font=(self.font_family, 11, "italic"), foreground="#7f8c8d", background=self.app_bg_color)
        
        self.style.configure("Accent.TButton", font=(self.font_family, 14, "bold"), padding=(15, 10), relief="flat",
                             foreground="white", background=self.primary_color, borderwidth=0)
        self.style.map("Accent.TButton",
            background=[('active', '#0056b3'), ('pressed', '#004085')],
            relief=[('pressed', 'sunken'), ('!pressed', 'flat')]
        )
        self.style.configure("Secondary.TButton", font=(self.font_family, 14), padding=(15,10), relief="flat",
                             foreground=self.text_color, background="#e9ecef", borderwidth=0)
        self.style.map("Secondary.TButton",
            background=[('active', '#ced4da'), ('pressed', '#adb5bd')]
        )

        app_frame = ttk.Frame(root, padding="25 25 25 25") 
        app_frame.pack(fill=tk.BOTH, expand=True)

        header_label = ttk.Label(app_frame, text="Vision Scribe", style="Header.TLabel")
        header_label.pack(pady=(10, 5))
        subheader_label = ttk.Label(app_frame, text="AI-Powered Image Caption Generator", style="SubHeader.TLabel")
        subheader_label.pack(pady=(0, 30))


        control_frame = ttk.Frame(app_frame, padding="15 0 15 0")
        control_frame.pack(fill=tk.X, pady=20)
        control_frame.columnconfigure(0, weight=1)
        control_frame.columnconfigure(1, weight=1)

        self.load_model_button = ttk.Button(control_frame, text="Load AI Model", style="Secondary.TButton", command=load_models_threaded)
        self.load_model_button.grid(row=0, column=0, padx=20, sticky="ew")

        self.upload_button = ttk.Button(control_frame, text="Upload Image", style="Accent.TButton", command=self.upload_image_action, state=tk.DISABLED)
        self.upload_button.grid(row=0, column=1, padx=20, sticky="ew")
        
        self.status_label = ttk.Label(app_frame, text="Welcome! Press 'Load AI Model' to start.", style="Status.TLabel", foreground=self.secondary_color)
        self.status_label.pack(pady=15)

        display_content_frame = ttk.Frame(app_frame)
        display_content_frame.pack(fill=tk.BOTH, expand=True, pady=10)
        display_content_frame.columnconfigure(0, weight=1) 
        display_content_frame.columnconfigure(1, weight=2) 
        display_content_frame.rowconfigure(0, weight=1)

        self.image_card = ttk.Frame(display_content_frame, style="Card.TFrame", padding=15)
        self.image_card.grid(row=0, column=0, padx=(0,15), pady=10, sticky="nsew")
        self.image_card.pack_propagate(False)

        self.image_label = ttk.Label(self.image_card, anchor="center", background=self.card_bg_color) 
        self.image_label.pack(fill=tk.BOTH, expand=True)
        self.update_image_placeholder(400,400) 

        self.caption_card = ttk.Frame(display_content_frame, style="Card.TFrame", padding=20)
        self.caption_card.grid(row=0, column=1, padx=(15,0), pady=10, sticky="nsew")
        self.caption_card.rowconfigure(0, weight=0) 
        self.caption_card.rowconfigure(1, weight=1) 
        self.caption_card.columnconfigure(0, weight=1)

        caption_title = ttk.Label(self.caption_card, text="AI Generated Caption:", font=(self.font_family, 18, "bold"), foreground=self.primary_color, background=self.card_bg_color)
        caption_title.grid(row=0, column=0, pady=(0,15), sticky="w")

        self.caption_text_widget = tk.Text(
            self.caption_card, wrap=tk.WORD, relief=tk.FLAT, borderwidth=0,
            font=(self.font_family, 16, "normal"), padx=5, pady=5,
            state=tk.DISABLED, bg=self.card_bg_color, fg=self.text_color,
            selectbackground="#a0c4ff", selectforeground=self.text_color 
        )
        self.caption_text_widget.grid(row=1, column=0, sticky="nsew")
        
        scrollbar = ttk.Scrollbar(self.caption_card, orient="vertical", command=self.caption_text_widget.yview, style="Vertical.TScrollbar")

        footer_frame = ttk.Frame(root, padding="15 10 15 10")
        footer_frame.pack(side=tk.BOTTOM, fill=tk.X)
        name_label = ttk.Label(footer_frame, text="Developed by Mayank Gour", style="Footer.TLabel", anchor="center")
        name_label.pack(expand=True)

        if not os.path.exists(MODEL_WEIGHTS_PATH) or not os.path.exists(VECTORIZER_PATH):
            self.update_status("Error: Model/vectorizer file missing. Check paths.", "red")
            self.load_model_button.config(state=tk.DISABLED)

    def update_image_placeholder(self, width, height):
        try:
            placeholder_font_size = int(min(width, height) / 15)
            font = ImageFont.truetype(self.font_family.lower().replace(" ", "") + ".ttf", placeholder_font_size) # Try to load font
        except IOError:
            font = ImageFont.load_default() # Fallback

        try:
            placeholder = Image.new('RGB', (int(width), int(height)), color = self.card_bg_color)
            draw = ImageDraw.Draw(placeholder)
            text = "🖼️ Your Image Here"
            
            bbox = draw.textbbox((0,0), text, font=font)
            text_width = bbox[2] - bbox[0]
            text_height = bbox[3] - bbox[1]
            
            x = (width - text_width) / 2
            y = (height - text_height) / 2
            draw.text((x, y), text, fill='#cccccc', font=font) 
            
            photo = ImageTk.PhotoImage(placeholder)
            self.image_label.config(image=photo)
            self.image_label.image = photo
        except Exception as e:
            print(f"Error creating placeholder: {e}")
            self.image_label.config(text="Image Preview", font=(self.font_family, 14))

    def update_status(self, message, color=None):
        if color is None: color = self.secondary_color
        if hasattr(self, 'status_label') and self.status_label.winfo_exists():
            self.root.after(0, lambda: self.status_label.config(text=message, foreground=color))

    def enable_upload_button(self):
         if hasattr(self, 'upload_button') and self.upload_button.winfo_exists():
            self.root.after(0, lambda: self.upload_button.config(state=tk.NORMAL))

    def display_image(self, file_path):
        global uploaded_image_bytes_global
        try:
            img_pil = Image.open(file_path)
            with open(file_path, 'rb') as f: uploaded_image_bytes_global = f.read()


            self.root.update_idletasks() 
            card_width = self.image_card.winfo_width()
            card_height = self.image_card.winfo_height()
            
            if card_width <= 1 or card_height <= 1: 
                card_width, card_height = 380, 380

            img_pil.thumbnail((card_width - 20, card_height - 20), Image.Resampling.LANCZOS) 
            
            photo = ImageTk.PhotoImage(img_pil)
            self.image_label.config(image=photo, text="")
            self.image_label.image = photo 
        except Exception as e:
            messagebox.showerror("Image Display Error", f"Could not display image: {e}")
            self.update_image_placeholder(self.image_card.winfo_width(), self.image_card.winfo_height())


    def upload_image_action(self): 
        global uploaded_image_path_global, uploaded_image_bytes_global
        if not caption_model_global or not vectorizer_global:
            messagebox.showwarning("Models Not Loaded", "Please load the AI model first.")
            return

        file_path = filedialog.askopenfilename(
            title="Select an Image File",
            filetypes=(("Image files", "*.jpg;*.jpeg;*.png"), ("All files", "*.*"))
        )
        if file_path:
            uploaded_image_path_global = file_path
            self.update_status(f"Image selected: {os.path.basename(file_path)}", self.primary_color)
            self.display_image(file_path)
            self.caption_text_widget.config(state=tk.NORMAL); self.caption_text_widget.delete(1.0, tk.END); self.caption_text_widget.config(state=tk.DISABLED)
            self.generate_and_display_caption()
        else:
            self.update_status("Image selection cancelled.", self.secondary_color)
            uploaded_image_bytes_global = None
            self.update_image_placeholder(self.image_card.winfo_width(), self.image_card.winfo_height())

    def generate_and_display_caption(self): 
        global uploaded_image_bytes_global
        if uploaded_image_bytes_global:
            self.update_status("Generating caption... Please wait.", self.primary_color)
            self.caption_text_widget.config(state=tk.NORMAL); self.caption_text_widget.delete(1.0, tk.END)
            self.caption_text_widget.insert(tk.END, "🤖 Thinking..."); self.caption_text_widget.config(state=tk.DISABLED)
            self.root.update_idletasks()
            try:
                threading.Thread(target=lambda: self.root.after(0, self._update_caption_widget, generate_caption_greedy_tk(uploaded_image_bytes_global)), daemon=True).start()
            except Exception as e:
                self._update_caption_widget(f"Error: {str(e)[:100]}")
        else:
            self.update_status("No image data. Upload an image.", self.secondary_color)
            
    def _update_caption_widget(self, caption_text): 
        self.caption_text_widget.config(state=tk.NORMAL); self.caption_text_widget.delete(1.0, tk.END)
        self.caption_text_widget.insert(tk.END, caption_text); self.caption_text_widget.config(state=tk.DISABLED)
        if "Error" not in caption_text and caption_text != "🤖 Thinking...":
            self.update_status("Caption generated successfully!", "green")
        elif "Error" in caption_text:
            self.update_status(f"{caption_text}", "red")


if __name__ == '__main__':
    root = tk.Tk()
    root_app = App(root)
    root.mainloop()

In [ ]:


import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk   # pip install pillow

class CaptionApp(ttk.Frame):
    PREVIEW_SIZE = 480

    def __init__(self, master):
        super().__init__(master)
        master.title("Image Captioning")
        master.geometry("960x720")
        master.minsize(800, 600)
        master.configure(bg="#222")          
        
        self.pack(fill="both", expand=True)


        style = ttk.Style(master)
        style.theme_use("clam")
        style.configure(".",          background="#222", foreground="#f5f5f5")
        style.configure("Header.TLabel",  font=("Segoe UI", 22, "bold"))
        style.configure("Caption.TEntry", font=("Segoe UI", 12))
        style.configure("TButton",        font=("Segoe UI", 11), padding=6)
        style.map("TButton",
                  background=[("active", "#444")],
                  foreground=[("disabled", "#888")])


        self.columnconfigure(0, weight=1)
        self.rowconfigure(1, weight=1)      
        


        ttk.Label(self, text="Image-to-Caption", style="Header.TLabel"
                 ).grid(row=0, column=0, pady=(20, 10))


        prev_frame = ttk.Frame(self, padding=4, style="Preview.TFrame")
        prev_frame.grid(row=1, column=0, pady=10)
        prev_frame.columnconfigure(0, weight=1)
        prev_frame.rowconfigure(0, weight=1)
        style.configure("Preview.TFrame", background="#2e2e2e")
        self.preview = tk.Label(prev_frame, bg="#2e2e2e",
                                width=self.PREVIEW_SIZE,
                                height=self.PREVIEW_SIZE)
        self.preview.grid(sticky="nsew")


        cap_frame = ttk.Frame(self, padding=(30, 10))
        cap_frame.grid(row=2, column=0, sticky="ew")
        cap_frame.columnconfigure(0, weight=1)
        ttk.Label(cap_frame, text="Caption:").grid(row=0, column=0, sticky="w")
        self.caption_var = tk.StringVar()
        ttk.Entry(cap_frame, textvariable=self.caption_var,
                  state="readonly", style="Caption.TEntry"
                 ).grid(row=1, column=0, sticky="ew", pady=4)


        btn_frame = ttk.Frame(self, padding=10)
        btn_frame.grid(row=3, column=0)
        ttk.Button(btn_frame, text="Open image", command=self.open_image
                  ).pack(side="left", padx=8)
        ttk.Button(btn_frame, text="Generate caption", command=self.gen_caption
                  ).pack(side="left", padx=8)


        ttk.Label(self, text="Made by Mayank Gour",
                  font=("Segoe UI", 10)
                 ).grid(row=4, column=0, pady=(10, 15))


        self.path   = None
        self.tk_img = None


    def open_image(self):
        ft = [("Images", "*.jpg *.jpeg *.png *.bmp *.gif"), ("All files", "*.*")]
        path = filedialog.askopenfilename(title="Choose an image", filetypes=ft)
        if not path:
            return
        self.path = path
        try:
            thumb = Image.open(path).convert("RGB")
            thumb.thumbnail((self.PREVIEW_SIZE, self.PREVIEW_SIZE))
            canvas = Image.new("RGB", (self.PREVIEW_SIZE, self.PREVIEW_SIZE),
                               color="#2e2e2e")
            x = (self.PREVIEW_SIZE - thumb.width) // 2
            y = (self.PREVIEW_SIZE - thumb.height) // 2
            canvas.paste(thumb, (x, y))
            self.tk_img = ImageTk.PhotoImage(canvas)
            self.preview.configure(image=self.tk_img)
            self.caption_var.set("")
        except Exception as e:
            messagebox.showerror("Error", str(e))
            self.path = None

    def gen_caption(self):
        if not self.path:
            messagebox.showinfo("Info", "Please select an image first.")
            return
        try:
            caption = greedy_algorithm(self.path)   
            self.caption_var.set(caption)
        except Exception as e:
            messagebox.showerror("Error", str(e))


if __name__ == "__main__":
    root = tk.Tk()
    CaptionApp(root)
    root.mainloop()
